In [0]:
raw_capture_path = (
    "abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/"
    "ns-trackpulse-dev/telemetry-events/"
)

display(dbutils.fs.ls(raw_capture_path))

path,name,size,modificationTime
abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/0/,0/,0,1792954842000
abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/1/,1/,0,1792961442000
abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/2/,2/,0,1792890042000
abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/3/,3/,0,1792957542000


In [0]:
items = dbutils.fs.ls(raw_capture_path)

display([
    {
        "name": item.name,
        "path": item.path,
        "size_bytes": item.size,
        "is_directory": item.isDir()
    }
    for item in items
])

is_directory,name,path,size_bytes
true,0/,abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/0/,0
true,1/,abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/1/,0
true,2/,abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/2/,0
true,3/,abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/3/,0


In [0]:
from pprint import pprint

for partition in dbutils.fs.ls(raw_capture_path):
    print(f"\nPartition: {partition.name}")
    try:
        for item in dbutils.fs.ls(partition.path):
            print({
                "name": item.name,
                "path": item.path,
                "size_bytes": item.size,
                "is_directory": item.isDir()
            })
    except Exception as e:
        print(f"Could not list partition: {e}")


Partition: 0/
{'name': '2026/', 'path': 'abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/0/2026/', 'size_bytes': 0, 'is_directory': True}

Partition: 1/
{'name': '2026/', 'path': 'abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/1/2026/', 'size_bytes': 0, 'is_directory': True}

Partition: 2/
{'name': '2026/', 'path': 'abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/2/2026/', 'size_bytes': 0, 'is_directory': True}

Partition: 3/
{'name': '2026/', 'path': 'abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/3/2026/', 'size_bytes': 0, 'is_directory': True}


In [0]:
# Purpose: Discover Event Hubs Capture files across all partitions.
# Safety: Read-only operation; does not modify data or checkpoints.

def find_capture_files(path, depth=0, max_depth=8):
    if depth > max_depth:
        return

    try:
        items = dbutils.fs.ls(path)

        for item in items:
            if item.isDir():
                find_capture_files(item.path, depth + 1, max_depth)
            else:
                print(
                    f"{item.name} | {item.size} bytes | {item.path}"
                )

    except Exception as e:
        print(f"ERROR: {path} | {e}")

for partition in dbutils.fs.ls(raw_capture_path):
    print(f"\n=== {partition.name} ===")
    find_capture_files(partition.path)


=== 0/ ===
41.avro | 2472 bytes | abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/0/2026/10/25/18/55/41.avro
41.avro | 4434671 bytes | abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/0/2026/10/25/20/55/41.avro

=== 1/ ===
41.avro | 5684 bytes | abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/1/2026/10/25/20/45/41.avro
41.avro | 4623049 bytes | abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/1/2026/10/25/20/55/41.avro
41.avro | 1676 bytes | abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/1/2026/10/28/01/35/41.avro

=== 2/ ===
41.avro | 4717 bytes | abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/2/2026/10/25/00/55/41.avro
41.avro | 760 bytes | abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net

In [0]:
# Purpose: Inspect one small Event Hubs Capture Avro file.
# Safety: Read-only; no Delta writes and no checkpoint changes.

sample_avro_path = (
    "abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/"
    "ns-trackpulse-dev/telemetry-events/3/2026/10/28/01/25/41.avro"
)

sample_df = (
    spark.read
    .format("avro")
    .load(sample_avro_path)
)

print("Capture file schema:")
sample_df.printSchema()

print("Sample records:")
display(sample_df.limit(5))

Capture file schema:
root
 |-- SequenceNumber: long (nullable = true)
 |-- Offset: string (nullable = true)
 |-- EnqueuedTimeUtc: string (nullable = true)
 |-- SystemProperties: map (nullable = true)
 |    |-- key: string
 |    |-- value: struct (valueContainsNull = true)
 |    |    |-- member0: long (nullable = true)
 |    |    |-- member1: double (nullable = true)
 |    |    |-- member2: string (nullable = true)
 |    |    |-- member3: binary (nullable = true)
 |-- Properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: struct (valueContainsNull = true)
 |    |    |-- member0: long (nullable = true)
 |    |    |-- member1: double (nullable = true)
 |    |    |-- member2: string (nullable = true)
 |    |    |-- member3: binary (nullable = true)
 |-- Body: binary (nullable = true)

Sample records:


SequenceNumber,Offset,EnqueuedTimeUtc,SystemProperties,Properties,Body
10510,8589934592,10/28/2026 1:27:43 AM,"Map(x-opt-enqueued-time -> List(1793150863905, null, null, null))",Map(),eyJldmVudF9pZCI6ICJURVNULWZhc3RwYXRoLXNtb2tlLTIwMjYxMDI4LTAyLWVuZ2luZSIsICJldmVudF90aW1lc3RhbXAiOiAiMjAyNi0xMC0yOFQwMToyNzo0MS45NjYxNTFaIiwgInZlaGljbGVfaWQiOiAiVkVILVRFU1QtMDAwMDEiLCAibGF0aXR1ZGUiOiAzMC4wNDQ0LCAibG9uZ2l0dWRlIjogMzEuMjM1NywgInJwbSI6IDE4MDAsICJmdWVsX3JhdGVfbHBoIjogMi41LCAiZW5naW5lX3RlbXBfYyI6IDExMi4wLCAiY2FyZ29fdGVtcF9jIjogMS4wLCAidGVzdF9tb2RlIjogdHJ1ZSwgInRlc3RfaWQiOiAiZmFzdHBhdGgtc21va2UtMjAyNjEwMjgtMDIifQ==
10511,8589934992,10/28/2026 1:27:43 AM,"Map(x-opt-enqueued-time -> List(1793150863905, null, null, null))",Map(),eyJldmVudF9pZCI6ICJURVNULWZhc3RwYXRoLXNtb2tlLTIwMjYxMDI4LTAyLWNhcmdvIiwgImV2ZW50X3RpbWVzdGFtcCI6ICIyMDI2LTEwLTI4VDAxOjI3OjQxLjk2NjE1MVoiLCAidmVoaWNsZV9pZCI6ICJWRUgtVEVTVC0wMDAwMiIsICJsYXRpdHVkZSI6IDMwLjA0NDQsICJsb25naXR1ZGUiOiAzMS4yMzU3LCAicnBtIjogMTUwMCwgImZ1ZWxfcmF0ZV9scGgiOiAxLjgsICJlbmdpbmVfdGVtcF9jIjogODUuMCwgImNhcmdvX3RlbXBfYyI6IDUuMCwgInRlc3RfbW9kZSI6IHRydWUsICJ0ZXN0X2lkIjogImZhc3RwYXRoLXNtb2tlLTIwMjYxMDI4LTAyIn0=
10512,8589935384,10/28/2026 1:27:43 AM,"Map(x-opt-enqueued-time -> List(1793150863905, null, null, null))",Map(),eyJldmVudF9pZCI6ICJURVNULWZhc3RwYXRoLXNtb2tlLTIwMjYxMDI4LTAyLW5vcm1hbCIsICJldmVudF90aW1lc3RhbXAiOiAiMjAyNi0xMC0yOFQwMToyNzo0MS45NjYxNTFaIiwgInZlaGljbGVfaWQiOiAiVkVILVRFU1QtMDAwMDMiLCAibGF0aXR1ZGUiOiAzMC4wNDQ0LCAibG9uZ2l0dWRlIjogMzEuMjM1NywgInJwbSI6IDEyMDAsICJmdWVsX3JhdGVfbHBoIjogMS4yLCAiZW5naW5lX3RlbXBfYyI6IDg1LjAsICJjYXJnb190ZW1wX2MiOiAxLjAsICJ0ZXN0X21vZGUiOiB0cnVlLCAidGVzdF9pZCI6ICJmYXN0cGF0aC1zbW9rZS0yMDI2MTAyOC0wMiJ9


In [0]:
# Purpose: Decode Event Hubs Capture binary payloads and inspect event metadata.
# Safety: Read-only sample inspection; no Delta writes or checkpoint changes.

from pyspark.sql.functions import col, decode

decoded_sample = (
    spark.read.format("avro").load(sample_avro_path)
    .select(
        "SequenceNumber",
        "EnqueuedTimeUtc",
        decode(col("Body"), "UTF-8").alias("event_json")
    )
)

display(decoded_sample.limit(5))

SequenceNumber,EnqueuedTimeUtc,event_json
10510,10/28/2026 1:27:43 AM,"{""event_id"": ""TEST-fastpath-smoke-20261028-02-engine"", ""event_timestamp"": ""2026-10-28T01:27:41.966151Z"", ""vehicle_id"": ""VEH-TEST-00001"", ""latitude"": 30.0444, ""longitude"": 31.2357, ""rpm"": 1800, ""fuel_rate_lph"": 2.5, ""engine_temp_c"": 112.0, ""cargo_temp_c"": 1.0, ""test_mode"": true, ""test_id"": ""fastpath-smoke-20261028-02""}"
10511,10/28/2026 1:27:43 AM,"{""event_id"": ""TEST-fastpath-smoke-20261028-02-cargo"", ""event_timestamp"": ""2026-10-28T01:27:41.966151Z"", ""vehicle_id"": ""VEH-TEST-00002"", ""latitude"": 30.0444, ""longitude"": 31.2357, ""rpm"": 1500, ""fuel_rate_lph"": 1.8, ""engine_temp_c"": 85.0, ""cargo_temp_c"": 5.0, ""test_mode"": true, ""test_id"": ""fastpath-smoke-20261028-02""}"
10512,10/28/2026 1:27:43 AM,"{""event_id"": ""TEST-fastpath-smoke-20261028-02-normal"", ""event_timestamp"": ""2026-10-28T01:27:41.966151Z"", ""vehicle_id"": ""VEH-TEST-00003"", ""latitude"": 30.0444, ""longitude"": 31.2357, ""rpm"": 1200, ""fuel_rate_lph"": 1.2, ""engine_temp_c"": 85.0, ""cargo_temp_c"": 1.0, ""test_mode"": true, ""test_id"": ""fastpath-smoke-20261028-02""}"


In [0]:
# Purpose: Inspect a larger Event Hubs Capture file to identify production events.
# Safety: Read-only; does not write Delta data or modify any checkpoint.

production_sample_path = (
    "abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/"
    "ns-trackpulse-dev/telemetry-events/0/2026/10/25/20/55/41.avro"
)

from pyspark.sql.functions import col, decode

production_sample_df = (
    spark.read.format("avro")
    .load(production_sample_path)
    .select(
        "SequenceNumber",
        "EnqueuedTimeUtc",
        decode(col("Body"), "UTF-8").alias("event_json")
    )
)

print("Records in this Capture file:", production_sample_df.count())
display(production_sample_df.limit(5))

Records in this Capture file: 10000


SequenceNumber,EnqueuedTimeUtc,event_json
10,10/25/2026 8:56:45 PM,"{""event_id"":""VEH-01501-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01501"",""latitude"":29.135393,""longitude"":31.108905,""speed_kph"":47.65,""odometer_km"":386492.417,""rpm"":2029,""fuel_rate_lph"":5.088,""engine_temp_c"":91.94,""ignition_on"":true,""trip_id"":""TRIP-01501-001"",""shipment_id"":""SHP-01501"",""driver_id"":""DRV-00421"",""route_id"":""RTE-007"",""cargo_temp_c"":null}"
11,10/25/2026 8:56:45 PM,"{""event_id"":""VEH-01502-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01502"",""latitude"":31.085828,""longitude"":29.822834,""speed_kph"":27.37,""odometer_km"":211264.352,""rpm"":1396,""fuel_rate_lph"":3.096,""engine_temp_c"":86.9,""ignition_on"":true,""trip_id"":""TRIP-01502-001"",""shipment_id"":""SHP-01502"",""driver_id"":""DRV-00421"",""route_id"":""RTE-003"",""cargo_temp_c"":1.93}"
12,10/25/2026 8:56:45 PM,"{""event_id"":""VEH-01503-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01503"",""latitude"":30.024133,""longitude"":31.33653,""speed_kph"":38.57,""odometer_km"":249726.579,""rpm"":1585,""fuel_rate_lph"":4.462,""engine_temp_c"":85.68,""ignition_on"":true,""trip_id"":""TRIP-01503-001"",""shipment_id"":""SHP-01503"",""driver_id"":""DRV-00421"",""route_id"":""RTE-001"",""cargo_temp_c"":-1.52}"
13,10/25/2026 8:56:45 PM,"{""event_id"":""VEH-01504-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01504"",""latitude"":30.023164,""longitude"":31.22895,""speed_kph"":35.57,""odometer_km"":300656.995,""rpm"":1734,""fuel_rate_lph"":3.396,""engine_temp_c"":108.03,""ignition_on"":true,""trip_id"":""TRIP-01504-001"",""shipment_id"":""SHP-01504"",""driver_id"":""DRV-00422"",""route_id"":""RTE-008"",""cargo_temp_c"":null}"
14,10/25/2026 8:56:45 PM,"{""event_id"":""VEH-01505-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01505"",""latitude"":29.743505,""longitude"":31.192282,""speed_kph"":18.44,""odometer_km"":403664.143,""rpm"":1217,""fuel_rate_lph"":2.301,""engine_temp_c"":93.59,""ignition_on"":true,""trip_id"":""TRIP-01505-001"",""shipment_id"":""SHP-01505"",""driver_id"":""DRV-00422"",""route_id"":""RTE-007"",""cargo_temp_c"":null}"


In [0]:
# Purpose: Define independent paths for streaming ingestion.
# Safety: Configuration only; does not read or write data.

raw_capture_path = (
    "abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/"
    "ns-trackpulse-dev/telemetry-events/"
)

bronze_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "bronze/telemetry_stream/"
)

stream_checkpoint_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "checkpoints/databricks_streaming/telemetry_capture/"
)

stream_schema_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "checkpoints/schemas/telemetry_capture/"
)

print("Raw source:", raw_capture_path)
print("Bronze target:", bronze_path)
print("Streaming checkpoint:", stream_checkpoint_path)
print("Auto Loader schema:", stream_schema_path)

Raw source: abfss://telemetry-raw@trackpulsestorage.dfs.core.windows.net/ns-trackpulse-dev/telemetry-events/
Bronze target: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/bronze/telemetry_stream/
Streaming checkpoint: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/databricks_streaming/telemetry_capture/
Auto Loader schema: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/checkpoints/schemas/telemetry_capture/


In [0]:
# Purpose: Incrementally discover Event Hubs Capture Avro files.
# Safety: Defines a streaming DataFrame only; does not start a streaming query.

from pyspark.sql.functions import col, decode, from_json, current_timestamp
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    LongType, BooleanType
)

event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("speed_kph", DoubleType(), True),
    StructField("odometer_km", DoubleType(), True),
    StructField("rpm", LongType(), True),
    StructField("fuel_rate_lph", DoubleType(), True),
    StructField("engine_temp_c", DoubleType(), True),
    StructField("ignition_on", BooleanType(), True),
    StructField("trip_id", StringType(), True),
    StructField("shipment_id", StringType(), True),
    StructField("driver_id", StringType(), True),
    StructField("route_id", StringType(), True),
    StructField("cargo_temp_c", DoubleType(), True)
])

capture_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "avro")
    .option("cloudFiles.schemaLocation", stream_schema_path)
    .load(raw_capture_path)
)

parsed_stream = (
    capture_stream
    .withColumn("event_json", decode(col("Body"), "UTF-8"))
    .withColumn("event", from_json(col("event_json"), event_schema))
    .select(
        col("SequenceNumber").alias("capture_sequence_number"),
        col("EnqueuedTimeUtc").alias("capture_enqueued_time_utc"),
        col("event.*"),
        col("event_json")
    )
)

print("Streaming source defined.")

Streaming source defined.


In [0]:
# Purpose: Exclude test events and append production telemetry to Bronze Delta.
# Safety: Uses a dedicated checkpoint; never resets or overwrites existing checkpoints.

from pyspark.sql.functions import coalesce, lit, to_timestamp

production_stream = (
    parsed_stream
    .filter(col("event_id").isNotNull())
    .filter(~col("event_id").startswith("TEST-"))
    .withColumn(
        "event_timestamp_utc",
        to_timestamp(col("event_timestamp"))
    )
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("source_format", lit("event_hubs_capture_avro"))
)

query = (
    production_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", stream_checkpoint_path)
    .trigger(availableNow=True)
    .start(bronze_path)
)

query.awaitTermination()
print("Available Capture files processed. Check query status and Bronze counts.")

Available Capture files processed. Check query status and Bronze counts.


In [0]:
# Purpose: Validate Bronze Delta ingestion and check for test events.
# Safety: Read-only validation; does not modify Delta data or checkpoints.

from pyspark.sql.functions import col

bronze_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
)

total_records = bronze_df.count()

test_records = (
    bronze_df
    .filter(col("event_id").startswith("TEST-"))
    .count()
)

print(f"Total Bronze records: {total_records:,}")
print(f"Test records found: {test_records:,}")
print("Bronze schema:")
bronze_df.printSchema()

display(bronze_df.limit(10))

Total Bronze records: 167,522
Test records found: 0
Bronze schema:
root
 |-- capture_sequence_number: long (nullable = true)
 |-- capture_enqueued_time_utc: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- speed_kph: double (nullable = true)
 |-- odometer_km: double (nullable = true)
 |-- rpm: long (nullable = true)
 |-- fuel_rate_lph: double (nullable = true)
 |-- engine_temp_c: double (nullable = true)
 |-- ignition_on: boolean (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- cargo_temp_c: double (nullable = true)
 |-- event_json: string (nullable = true)
 |-- event_timestamp_utc: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 

capture_sequence_number,capture_enqueued_time_utc,event_id,event_timestamp,vehicle_id,latitude,longitude,speed_kph,odometer_km,rpm,fuel_rate_lph,engine_temp_c,ignition_on,trip_id,shipment_id,driver_id,route_id,cargo_temp_c,event_json,event_timestamp_utc,bronze_ingested_at,source_format
10,10/25/2026 8:56:44 PM,VEH-01001-0002000,2026-10-02T09:19:02Z,VEH-01001,29.916718,32.165056,50.1,339756.994,1997,4.948,94.34,true,TRIP-01001-001,SHP-01001,DRV-00281,RTE-005,-0.61,"{""event_id"":""VEH-01001-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01001"",""latitude"":29.916718,""longitude"":32.165056,""speed_kph"":50.1,""odometer_km"":339756.994,""rpm"":1997,""fuel_rate_lph"":4.948,""engine_temp_c"":94.34,""ignition_on"":true,""trip_id"":""TRIP-01001-001"",""shipment_id"":""SHP-01001"",""driver_id"":""DRV-00281"",""route_id"":""RTE-005"",""cargo_temp_c"":-0.61}",2026-10-02T09:19:02.000Z,2026-10-28T13:36:52.241Z,event_hubs_capture_avro
11,10/25/2026 8:56:44 PM,VEH-01002-0002000,2026-10-02T09:19:02Z,VEH-01002,29.329311,30.83815,42.08,223419.276,1927,8.331,93.25,true,TRIP-01002-001,SHP-01002,DRV-00281,RTE-006,null,"{""event_id"":""VEH-01002-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01002"",""latitude"":29.329311,""longitude"":30.83815,""speed_kph"":42.08,""odometer_km"":223419.276,""rpm"":1927,""fuel_rate_lph"":8.331,""engine_temp_c"":93.25,""ignition_on"":true,""trip_id"":""TRIP-01002-001"",""shipment_id"":""SHP-01002"",""driver_id"":""DRV-00281"",""route_id"":""RTE-006"",""cargo_temp_c"":null}",2026-10-02T09:19:02.000Z,2026-10-28T13:36:52.241Z,event_hubs_capture_avro
12,10/25/2026 8:56:44 PM,VEH-01003-0002000,2026-10-02T09:19:02Z,VEH-01003,30.067355,31.238801,38.16,437581.221,1745,3.737,93.6,true,TRIP-01003-001,SHP-01003,DRV-00281,RTE-010,null,"{""event_id"":""VEH-01003-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01003"",""latitude"":30.067355,""longitude"":31.238801,""speed_kph"":38.16,""odometer_km"":437581.221,""rpm"":1745,""fuel_rate_lph"":3.737,""engine_temp_c"":93.6,""ignition_on"":true,""trip_id"":""TRIP-01003-001"",""shipment_id"":""SHP-01003"",""driver_id"":""DRV-00281"",""route_id"":""RTE-010"",""cargo_temp_c"":null}",2026-10-02T09:19:02.000Z,2026-10-28T13:36:52.241Z,event_hubs_capture_avro
13,10/25/2026 8:56:44 PM,VEH-01004-0002000,2026-10-02T09:19:02Z,VEH-01004,29.983199,31.137029,29.28,272599.3,1509,4.087,86.12,true,TRIP-01004-001,SHP-01004,DRV-00282,RTE-001,-0.27,"{""event_id"":""VEH-01004-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01004"",""latitude"":29.983199,""longitude"":31.137029,""speed_kph"":29.28,""odometer_km"":272599.3,""rpm"":1509,""fuel_rate_lph"":4.087,""engine_temp_c"":86.12,""ignition_on"":true,""trip_id"":""TRIP-01004-001"",""shipment_id"":""SHP-01004"",""driver_id"":""DRV-00282"",""route_id"":""RTE-001"",""cargo_temp_c"":-0.27}",2026-10-02T09:19:02.000Z,2026-10-28T13:36:52.241Z,event_hubs_capture_avro
14,10/25/2026 8:56:44 PM,VEH-01005-0002000,2026-10-02T09:19:02Z,VEH-01005,30.199356,31.225277,40.13,144691.056,1904,7.588,90.21,true,TRIP-01005-001,SHP-01005,DRV-00282,RTE-010,null,"{""event_id"":""VEH-01005-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01005"",""latitude"":30.199356,""longitude"":31.225277,""speed_kph"":40.13,""odometer_km"":144691.056,""rpm"":1904,""fuel_rate_lph"":7.588,""engine_temp_c"":90.21,""ignition_on"":true,""trip_id"":""TRIP-01005-001"",""shipment_id"":""SHP-01005"",""driver_id"":""DRV-00282"",""route_id"":""RTE-010"",""cargo_temp_c"":null}",2026-10-02T09:19:02.000Z,2026-10-28T13:36:52.241Z,event_hubs_capture_avro
15,10/25/2026 8:56:44 PM,VEH-01006-0002000,2026-10-02T09:19:02Z,VEH-01006,30.027134,31.219993,50.61,298305.964,2082,4.315,90.72,true,TRIP-01006-001,SHP-01006,DRV-00282,RTE-006,null,"{""event_id"":""VEH-01006-0002000"",""event_timestamp"":""2026-10-02T09:19:02Z"",""vehicle_id"":""VEH-01006"",""latitude"":30.027134,""lo

In [0]:
# Purpose: Profile Bronze data quality before creating the Silver layer.
# Safety: Read-only checks; does not modify Bronze, Silver, or checkpoints.

from pyspark.sql.functions import col, count, countDistinct, sum as spark_sum, to_timestamp

bronze_df = spark.read.format("delta").load(bronze_path)

print("=== Record counts ===")
print(f"Total records: {bronze_df.count():,}")
print(f"Distinct event IDs: {bronze_df.select('event_id').distinct().count():,}")

print("\n=== Missing required fields ===")
bronze_df.select(
    spark_sum(col("event_id").isNull().cast("int")).alias("missing_event_id"),
    spark_sum(col("vehicle_id").isNull().cast("int")).alias("missing_vehicle_id"),
    spark_sum(col("event_timestamp").isNull().cast("int")).alias("missing_event_timestamp")
).show()

print("\n=== Duplicate event IDs ===")
(
    bronze_df.groupBy("event_id")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
    .show(20, truncate=False)
)

print("\n=== Event time range ===")
bronze_df.select(
    to_timestamp("event_timestamp").alias("event_time")
).selectExpr(
    "min(event_time) AS earliest_event",
    "max(event_time) AS latest_event",
    "count_if(event_time IS NULL) AS invalid_event_timestamps"
).show(truncate=False)

=== Record counts ===
Total records: 41,522
Distinct event IDs: 41,522

=== Missing required fields ===
+----------------+------------------+-----------------------+
|missing_event_id|missing_vehicle_id|missing_event_timestamp|
+----------------+------------------+-----------------------+
|               0|                12|                     12|
+----------------+------------------+-----------------------+


=== Duplicate event IDs ===
+--------+-----+
|event_id|count|
+--------+-----+
+--------+-----+


=== Event time range ===
+-------------------+-------------------+------------------------+
|earliest_event     |latest_event       |invalid_event_timestamps|
+-------------------+-------------------+------------------------+
|2026-10-02 09:19:02|2026-10-24 23:12:54|12                      |
+-------------------+-------------------+------------------------+



In [0]:
# Purpose: Load the existing Bronze Delta table and define data quality rules.
# Safety: Read-only; does not modify Bronze data or any checkpoint.

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, DoubleType, LongType, BooleanType, TimestampType
)

bronze_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "bronze/telemetry_stream/"
)

bronze_df = spark.read.format("delta").load(bronze_path)

expected_schema = {
    "event_id": StringType,
    "event_timestamp": StringType,
    "vehicle_id": StringType,
    "latitude": DoubleType,
    "longitude": DoubleType,
    "speed_kph": DoubleType,
    "odometer_km": DoubleType,
    "rpm": LongType,
    "fuel_rate_lph": DoubleType,
    "engine_temp_c": DoubleType,
    "ignition_on": BooleanType,
    "trip_id": StringType,
    "shipment_id": StringType,
    "driver_id": StringType,
    "route_id": StringType,
    "cargo_temp_c": DoubleType,
}

print(f"Bronze record count: {bronze_df.count():,}")
print("Loaded Bronze schema and validation rules.")

Bronze record count: 41,522
Loaded Bronze schema and validation rules.


In [0]:
# Purpose: Verify that required telemetry columns exist and their data types match.
# Safety: Read-only schema validation.

actual_types = {
    field.name: type(field.dataType)
    for field in bronze_df.schema.fields
}

schema_results = []

for column_name, expected_type in expected_schema.items():
    if column_name not in actual_types:
        status = "FAIL"
        actual_type = "MISSING"
    else:
        actual_type = actual_types[column_name].__name__
        status = (
            "PASS"
            if actual_types[column_name] == expected_type
            else "FAIL"
        )

    schema_results.append(
        (column_name, expected_type.__name__, actual_type, status)
    )

schema_report = spark.createDataFrame(
    schema_results,
    ["column_name", "expected_type", "actual_type", "status"]
)

display(schema_report)

failed_schema_checks = sum(
    1 for result in schema_results if result[3] == "FAIL"
)

print(f"Schema checks failed: {failed_schema_checks}")

column_name,expected_type,actual_type,status
event_id,StringType,StringType,PASS
event_timestamp,StringType,StringType,PASS
vehicle_id,StringType,StringType,PASS
latitude,DoubleType,DoubleType,PASS
longitude,DoubleType,DoubleType,PASS
speed_kph,DoubleType,DoubleType,PASS
odometer_km,DoubleType,DoubleType,PASS
rpm,LongType,LongType,PASS
fuel_rate_lph,DoubleType,DoubleType,PASS
engine_temp_c,DoubleType,DoubleType,PASS


Schema checks failed: 0


In [0]:
# Purpose: Validate required fields, timestamps, GPS coordinates, and numeric ranges.
# Safety: Read-only; adds temporary validation flags without changing Bronze.

validated_df = (
    bronze_df
    .withColumn(
        "parsed_event_timestamp",
        F.to_timestamp("event_timestamp")
    )
    .withColumn(
        "dq_missing_event_id",
        F.col("event_id").isNull() | (F.trim("event_id") == "")
    )
    .withColumn(
        "dq_missing_vehicle_id",
        F.col("vehicle_id").isNull() | (F.trim("vehicle_id") == "")
    )
    .withColumn(
        "dq_invalid_timestamp",
        F.col("parsed_event_timestamp").isNull()
    )
    .withColumn(
        "dq_invalid_latitude",
        F.col("latitude").isNotNull()
        & ~F.col("latitude").between(-90, 90)
    )
    .withColumn(
        "dq_invalid_longitude",
        F.col("longitude").isNotNull()
        & ~F.col("longitude").between(-180, 180)
    )
    .withColumn(
        "dq_negative_speed",
        F.col("speed_kph").isNotNull() & (F.col("speed_kph") < 0)
    )
    .withColumn(
        "dq_negative_odometer",
        F.col("odometer_km").isNotNull() & (F.col("odometer_km") < 0)
    )
    .withColumn(
        "dq_negative_rpm",
        F.col("rpm").isNotNull() & (F.col("rpm") < 0)
    )
    .withColumn(
        "dq_negative_fuel_rate",
        F.col("fuel_rate_lph").isNotNull() & (F.col("fuel_rate_lph") < 0)
    )
)

dq_flags = [
    "dq_missing_event_id",
    "dq_missing_vehicle_id",
    "dq_invalid_timestamp",
    "dq_invalid_latitude",
    "dq_invalid_longitude",
    "dq_negative_speed",
    "dq_negative_odometer",
    "dq_negative_rpm",
    "dq_negative_fuel_rate",
]

dq_summary = []

for flag in dq_flags:
    dq_summary.append(
        (
            flag,
            validated_df.filter(F.col(flag)).count()
        )
    )

dq_summary_df = spark.createDataFrame(
    dq_summary,
    ["check_name", "failed_records"]
)

display(dq_summary_df)

check_name,failed_records
dq_missing_event_id,0
dq_missing_vehicle_id,12
dq_invalid_timestamp,12
dq_invalid_latitude,0
dq_invalid_longitude,0
dq_negative_speed,0
dq_negative_odometer,0
dq_negative_rpm,0
dq_negative_fuel_rate,0


In [0]:
# Purpose: Check event uniqueness, exclude test events from production eligibility,
# and calculate the overall Data Quality Gate result.
# Safety: Read-only; does not delete records or write Silver.

total_records = validated_df.count()

duplicate_event_ids = (
    validated_df
    .groupBy("event_id")
    .count()
    .filter(F.col("event_id").isNotNull() & (F.col("count") > 1))
    .count()
)

test_events = validated_df.filter(
    F.coalesce(F.col("test_mode").cast("boolean"), F.lit(False))
    | F.coalesce(F.col("event_id").startswith("TEST-"), F.lit(False))
    | F.col("test_id").isNotNull()
).count() if {"test_mode", "test_id"}.issubset(set(validated_df.columns)) else (
    validated_df
    .filter(F.col("event_id").startswith("TEST-"))
    .count()
)

invalid_required = (
    F.col("dq_missing_event_id")
    | F.col("dq_missing_vehicle_id")
    | F.col("dq_invalid_timestamp")
)

eligible_df = validated_df.filter(
    ~invalid_required
    & ~F.col("dq_invalid_latitude")
    & ~F.col("dq_invalid_longitude")
    & ~F.col("dq_negative_speed")
    & ~F.col("dq_negative_odometer")
    & ~F.col("dq_negative_rpm")
    & ~F.col("dq_negative_fuel_rate")
    & ~F.col("event_id").startswith("TEST-")
)

eligible_count = eligible_df.count()

print(f"Total Bronze records: {total_records:,}")
print(f"Duplicate event IDs: {duplicate_event_ids:,}")
print(f"Test events detected: {test_events:,}")
print(f"Eligible for Silver: {eligible_count:,}")
print(f"Rejected / quarantined: {total_records - eligible_count:,}")
print(f"Schema checks failed: {failed_schema_checks}")

gate_passed = (
    failed_schema_checks == 0
    and duplicate_event_ids == 0
)

print(f"Schema + uniqueness gate passed: {gate_passed}")

Total Bronze records: 41,522
Duplicate event IDs: 0
Test events detected: 0
Eligible for Silver: 41,510
Rejected / quarantined: 12
Schema checks failed: 0
Schema + uniqueness gate passed: True


In [0]:

checkpoint_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "checkpoints/databricks_streaming/telemetry_capture/"
)

try:
    for item in dbutils.fs.ls(checkpoint_path):
        print(item.name, item.size)
except Exception as e:
    print("ERROR:", str(e)[:500])


__tmp_path_dir/ 0
commits/ 0
metadata 45
offsets/ 0
sources/ 0
